# MiniCells — HF Checkpoints / GitHub Results

CPU engineering acceptance only. **Accelerator: None.** No R3 rerun or R3b training.
Enable the existing Kaggle Secrets **HF_TOKEN** and **GITHUB_TOKEN**.
No Add Input, local downloads, manual hash fields or Kaggle Output dependency.

- **A:** defaults → Run All. Auto preflight → synthetic 0→6 → HF persistence + GitHub index.
- **B:** new cloud Session, same source/run/repo; set `SEGMENT_END_STEP=10`. Run All → remote restore 6→10.
- **C:** new cloud Session; set `ACTION="evaluate"`. Optionally paste the printed `checkpoint_index_url` into `GITHUB_INDEX_URL` to verify GitHub→HF recovery. Run All evaluates historical step6 and compares B with a separate continuous CPU 0→10 control.

Save each version's logs. No unit suite, GPU Pilot or real dataset is executed here.
Real cross-session acceptance remains NOT_VERIFIED until those version logs and HF revisions are reviewed.


In [ ]:
# The requested dedicated repo; a bare slug resolves to HF_TOKEN's personal account.
HF_REPO = "minicells-research-checkpoints"
CREATE_PRIVATE_REPO = True  # create this dedicated repo if absent; never change existing visibility
ACTION = "auto"  # auto | train | resume | evaluate | storage-preflight | publish-only
SEGMENT_END_STEP = 6  # A:6, B:10; execution boundary, not a research-budget change
PUBLISH_GITHUB = True
GITHUB_INDEX_URL = None  # C: printed checkpoint_index_url, optional


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

SOURCE_SHA = "660c0abcfcc137ccf423dd92e27c47d3746c6e96"
CODE = Path("/kaggle/working/minicells-hf-code-" + SOURCE_SHA[:12])
WORK = Path("/kaggle/working/minicells-hf-engineering")
ENTRY = CODE / "research/projects/cellular-language-models/engineering/hf-checkpoint-storage/run.py"
os.environ.update(OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1")

def execute(args, cwd=None):
    print("[HF Engineering] " + " ".join(map(str,args)), flush=True)
    return subprocess.run(list(map(str,args)), cwd=cwd, check=True)

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in (("HF_TOKEN", "GITHUB_TOKEN") if PUBLISH_GITHUB else ("HF_TOKEN",)):
    if not os.environ.get(name, "").strip():
        try:
            os.environ[name] = secrets.get_secret(name).strip()
        except Exception:
            raise RuntimeError("Enable the existing Kaggle Secret " + name + " on this notebook") from None
    if not os.environ[name]:
        raise RuntimeError("Missing Kaggle Secret " + name)

if not (CODE / ".git").exists():
    CODE.mkdir(parents=True, exist_ok=False)
    execute(["git", "init", "--quiet", CODE])
    execute(["git", "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git"], CODE)
    execute(["git", "sparse-checkout", "init", "--no-cone"], CODE)
    execute(["git", "sparse-checkout", "set", "--no-cone", "/pyproject.toml", "/src/", "/shared/python/",
             "/research/projects/cellular-language-models/engineering/",
             "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
             "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
             "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
             "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
             "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/"], CODE)
    execute(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_SHA], CODE)
    execute(["git", "checkout", "--detach", "FETCH_HEAD"], CODE)
if subprocess.check_output(["git", "rev-parse", "HEAD"],cwd=CODE,text=True).strip() != SOURCE_SHA:
    raise RuntimeError("SOURCE_SHA_MISMATCH")

sys.path.insert(0,str(CODE / "shared/python"))
from minicells_artifacts.hf_storage import HubBackend
try:
    backend = HubBackend(HF_REPO)
    if CREATE_PRIVATE_REPO:
        backend.api.create_repo(repo_id=backend.repo, repo_type="model", private=True, exist_ok=True)
    print(json.dumps({"hf_repo":backend.repo,"source_sha":SOURCE_SHA,"scope":"SYNTHETIC_CPU_ENGINEERING_ONLY"}), flush=True)
except Exception as error:
    # Error messages can include signed URLs; log only the class.
    raise RuntimeError("HF repository setup failed (" + type(error).__name__ + "). Check private repo/token permissions.") from None

output = WORK if ACTION != "evaluate" else WORK.with_name(WORK.name + "-evaluation")
args = [sys.executable, ENTRY, "--mode", ACTION, "--hf-repo", backend.repo,
        "--source-sha", SOURCE_SHA, "--output", output, "--segment-end-step", str(SEGMENT_END_STEP)]
if PUBLISH_GITHUB:
    args += ["--publish-github"]
if GITHUB_INDEX_URL:
    args += ["--github-index-url", GITHUB_INDEX_URL]
execute(args, CODE)
print("HF_CROSS_SESSION_RESUME=NOT_VERIFIED until A/B/C revisions and state comparison are reviewed.", flush=True)
